# Ad-level Threshold Tuning Notebook
- Load Ad-level table
- Explore distribution of key metrics
- Visualize natural clusters
- Define and tune thresholds for Local v National
- Keep Addressable as a separate concept

Key Features used:
- Covereage Score
- Entropy (Normalized)
- Significant DMAs
- DMA Mix Ratios


In [0]:
!pip install --upgrade matplotlib
%restart_python


In [0]:
import pandas as pd
import numpy as np
from pyspark.sql import functions as F
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.isotonic import IsotonicRegression

plt.rcParams["figure.figsize"] = (8, 6)
plt.rcParams["axes.grid"] = True
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

In [0]:
df = spark.table("dev.mohit_gangwani.ad_labeling_final_features_unbinned_012126").toPandas()
df.fillna(0, inplace=True)
df.info()

In [0]:
df.rename(
    columns={
        "top5_dma_mix_ratio": "mix_ratio",
        "significant_dma_count_05": "sig_dma_count",
        "significant_region_count_05": "sig_region_count",
        'top_region_mix': 'region_mix'
    },
    inplace=True,
)
df.describe()

In [0]:
metrics = [
    "coverage_score",
    "entropy_norm",
    "sig_dma_count",
    "mix_ratio",
    "dma_in_90",
    "region_count_90",
    "sig_region_count",
    "region_mix",
    'region_count_90'
]
for col in metrics:
    df.loc[:, col] = df[col].astype(float)

df.head(20).display()

In [0]:
ndf = df[df["total_impressions"] >= 1000].copy()
ndf.reset_index(drop=True, inplace=True)

In [0]:
ndf.loc[:, 'norm_reg_mix'] = (ndf["region_mix"] - 0.167) / (ndf["region_mix"].max() - 0.167)
ndf.loc[:, 'norm_reg_mix'] = np.where(ndf.norm_reg_mix < 0, 0, ndf.norm_reg_mix)
df.loc[:, 'norm_reg_mix'] = (df["region_mix"] - 0.167) / (df["region_mix"].max() - 0.167)
df.loc[:, 'norm_reg_mix'] = np.where(df.norm_reg_mix < 0, 0, df.norm_reg_mix)

In [0]:
metrics = [
    "coverage_score",
    "entropy_norm",
    "sig_dma_count",
    "mix_ratio",
    "dma_in_90",
    "region_count_90",
    "sig_region_count",
    "norm_reg_mix",
]

In [0]:
def create_hist(df, metrics):
    for col in metrics:
        plt.figure()
        plt.hist(df[col], bins=50)
        # plt.yscale("log")
        plt.title(f"Distribution of {col.replace('_', ' ').title()}")
        plt.xlabel(col)
        plt.ylabel("Count")
        plt.show()

In [0]:
# create_hist(df, metrics)

In [0]:
create_hist(ndf, metrics)

In [0]:
qs = [i/20 for i in range(1, 20)]
qs.append(0.99)
qs.insert(0, 0.01)
quantiles = ndf[metrics].quantile(qs)
quantiles.index = qs
quantiles.reset_index(inplace=True)
display(quantiles)

In [0]:
def plot_a_vs_b(df_plot, a, b, a_name, b_name):
    plt.figure(figsize=(8, 7))

    # for t in df_plot['ad_type'].unique():
    #     subset = df_plot[df_plot['ad_type'] == t]
    plt.scatter(
        df_plot[a],
        df_plot[b],
        alpha=0.25,
        s=2,
    )

    plt.ylabel(b_name)
    plt.xlabel(a_name)
    plt.title(f'{a_name} vs {b_name}')
    plt.legend()
    plt.show()

In [0]:
plot_dict = {
    "coverage_score": "Coverage Score",
    "entropy_norm": "Entropy Norm",
    "mix_ratio": "Top 5 DMA Mix Ratio",
    "sig_dma_count": "Significant DMAs",
    "dma_in_90": "# of DMAs in 90 percent of Ads",
    "region_count_90": "# of Regions in 90 percent of Ads",
    'sig_region_count': "Significant Regions",
    'region_mix': "Top Region Mix Ratio",
}
done = []
for a, v1 in plot_dict.items():
    for b, v2 in plot_dict.items():
        if a != b and sorted([a, b]) not in done:
            done.append(sorted([a, b]))
            plot_a_vs_b(
                df_plot=ndf,
                a=a,
                b=b,
                a_name=v1,
                b_name=v2,
            )

In [0]:
class LocalNationalCalibrator:
    """
    Local vs National classifier with:
      1) Localness score (monotonic, interpretable)
      2) Anchor-based isotonic calibration -> P(local)
      3) Edge-case overrides
      4) High-precision mixed ads resolver to reduce mixed ads rate <10%

    Required columns in input df:
      - ad_id
      - total_impressions
      - coverage_score
      - entropy_norm
      - mix_ratio
      - dma_in_90
      - region_count_90
      - norm_reg_mix
    """

    def __init__(
        self,
        # score weights (sum to 1)
        w=None,
        # anchor seeds for calibration
        seed_lo=0.15,
        seed_hi=0.85,
        # final classification thresholds on calibrated probability
        p_lo=0.15,
        p_hi=0.85,
        # local degenerate override guardrail
        min_imps_local_override=20,
        # big DMA corner targeting override (national)
        corner_entropy_max=0.40,
        corner_coverage_max=0.10,
        corner_region_min=4,     # use 3 if you have ~8 regions; 4 if ~10
        # mixed ads resolver strict rules
        resolver_dma90_local_max=5,
        resolver_region90_local_max=2,
        resolver_region90_national_min=5,
        resolver_dma90_national_min=90,
        resolver_cov_national_min=0.72,
        resolver_ent_national_min=0.83,
        # calibration fit speed guardrail
        sample_cap=500_000,
        random_state=42,
    ):
        self.w = w or {"mix": 0.35, "ent": 0.35, "cov": 0.20, "reg": 0.10}

        self.seed_lo = seed_lo
        self.seed_hi = seed_hi
        self.p_lo = p_lo
        self.p_hi = p_hi

        self.min_imps_local_override = min_imps_local_override

        self.corner_entropy_max = corner_entropy_max
        self.corner_coverage_max = corner_coverage_max
        self.corner_region_min = corner_region_min

        self.resolver_dma90_local_max = resolver_dma90_local_max
        self.resolver_region90_local_max = resolver_region90_local_max

        self.resolver_region90_national_min = resolver_region90_national_min
        self.resolver_dma90_national_min = resolver_dma90_national_min
        self.resolver_cov_national_min = resolver_cov_national_min
        self.resolver_ent_national_min = resolver_ent_national_min

        self.sample_cap = sample_cap
        self.random_state = random_state

        self.iso = None

    # ----------------------------
    # Utilities
    # ----------------------------
    @staticmethod
    def _to_float(s):
        return pd.to_numeric(s, errors="coerce").astype(float)

    @staticmethod
    def _to_int(s):
        return pd.to_numeric(s, errors="coerce").fillna(np.nan).astype("Int64")

    def validate_columns(self, df: pd.DataFrame):
        required = {
            "ad_id",
            "total_impressions",
            "coverage_score",
            "entropy_norm",
            "mix_ratio",
            "dma_in_90",
            "region_count_90",
            "norm_reg_mix",
        }
        missing = sorted(list(required - set(df.columns)))
        if missing:
            raise ValueError(f"Missing required columns: {missing}")

    # ----------------------------
    # Localness score
    # ----------------------------
    def compute_localness(self, df: pd.DataFrame) -> pd.Series:
        cov = self._to_float(df["coverage_score"]).fillna(0.0).clip(0, 1)
        ent = self._to_float(df["entropy_norm"]).fillna(1.0).clip(0, 1)
        mix = self._to_float(df["mix_ratio"]).fillna(0.0).clip(0, 1)
        reg = self._to_float(df["norm_reg_mix"]).fillna(1.0).clip(0, 1)

        s = (
            self.w["mix"] * mix
            + self.w["ent"] * (1 - ent)
            + self.w["cov"] * (1 - cov)
            + self.w["reg"] * reg
        )
        return s.clip(0, 1)

    # ----------------------------
    # Calibration (anchors -> isotonic)
    # ----------------------------
    def fit_isotonic(self, df: pd.DataFrame, score_col="localness"):
        x = self._to_float(df[score_col]).fillna(0.5).clip(0, 1).to_numpy()
        y = np.full(len(df), np.nan)

        y[x <= self.seed_lo] = 0  # national anchors
        y[x >= self.seed_hi] = 1  # local anchors

        m = ~np.isnan(y)
        X, Y = x[m], y[m].astype(int)

        if len(X) == 0:
            raise RuntimeError("No anchors found. Adjust seed_lo/seed_hi.")

        rng = np.random.default_rng(self.random_state)
        if len(X) > self.sample_cap:
            idx = rng.choice(len(X), size=self.sample_cap, replace=False)
            X, Y = X[idx], Y[idx]

        self.iso = IsotonicRegression(y_min=0.0, y_max=1.0, increasing=True, out_of_bounds="clip")
        self.iso.fit(X, Y)

    # ----------------------------
    # Overrides
    # ----------------------------
    def apply_overrides(self, df: pd.DataFrame) -> pd.DataFrame:
        out = df.copy()

        imps = self._to_float(out["total_impressions"]).fillna(0.0)
        dma90 = self._to_int(out["dma_in_90"]).fillna(999).astype(int)
        reg90 = self._to_int(out["region_count_90"]).fillna(999).astype(int)

        ent = self._to_float(out["entropy_norm"]).fillna(1.0).clip(0, 1)
        cov = self._to_float(out["coverage_score"]).fillna(1.0).clip(0, 1)
        mix = self._to_float(out["mix_ratio"]).fillna(0.0).clip(0, 1)

        # Local: 90% in 1 DMA (with floor)
        local_dma90_1 = (dma90 == 1) & (imps >= self.min_imps_local_override)

        # Local: 90% in 1 region AND very concentrated (guarded)
        local_reg90_1 = (reg90 == 1) & (mix >= 0.60) & (imps >= self.min_imps_local_override)

        # National: big-DMA corner targeting pattern
        corner_national = (ent <= self.corner_entropy_max) & (cov <= self.corner_coverage_max) & (reg90 >= self.corner_region_min)

        # Apply probability clamps
        out.loc[local_dma90_1, "p_local_cal"] = np.maximum(out.loc[local_dma90_1, "p_local_cal"], 0.95)
        out.loc[local_reg90_1, "p_local_cal"] = np.maximum(out.loc[local_reg90_1, "p_local_cal"], 0.90)
        out.loc[corner_national, "p_local_cal"] = np.minimum(out.loc[corner_national, "p_local_cal"], 0.05)

        # Reasons
        out["override_reason"] = out.get("override_reason", "").astype(str)
        out.loc[local_dma90_1, "override_reason"] = self._append_reason(out.loc[local_dma90_1, "override_reason"], "local_dma_in_90_eq_1")
        out.loc[local_reg90_1, "override_reason"] = self._append_reason(out.loc[local_reg90_1, "override_reason"], "local_region_count_90_eq_1_guarded")
        out.loc[corner_national, "override_reason"] = self._append_reason(out.loc[corner_national, "override_reason"], "national_big_dma_corner")

        out["p_national_cal"] = 1 - out["p_local_cal"]
        return out

    @staticmethod
    def _append_reason(series: pd.Series, reason: str) -> pd.Series:
        s = series.fillna("").astype(str)
        return np.where(s.eq("") | s.eq("nan"), reason, s + "," + reason)

    # ----------------------------
    # mixed ads resolver (high precision)
    # ----------------------------
    def resolve_borderline(self, df: pd.DataFrame) -> pd.DataFrame:
        res = df.copy()

        m_border = res["label"].eq("mixed")

        imps = self._to_float(res["total_impressions"]).fillna(0.0)
        dma90 = self._to_int(res["dma_in_90"]).fillna(999).astype(int)
        reg90 = self._to_int(res["region_count_90"]).fillna(999).astype(int)

        ent = self._to_float(res["entropy_norm"]).fillna(1.0).clip(0, 1)
        cov = self._to_float(res["coverage_score"]).fillna(0.0).clip(0, 1)
        mix = self._to_float(res["mix_ratio"]).fillna(0.0).clip(0, 1)

        # Local resolver (strict): concentrated in few DMAs + few regions
        local_resolver = (
            (dma90 <= self.resolver_dma90_local_max) &
            (reg90 <= self.resolver_region90_local_max) &
            (imps >= self.min_imps_local_override)
        )
        # Optional extra precision (uncomment if needed):
        # local_resolver &= (mix >= 0.60)

        # National resolver (strict): wide region spread OR many DMAs OR wide+high entropy
        national_resolver = (
            (reg90 >= self.resolver_region90_national_min) |
            (dma90 >= self.resolver_dma90_national_min) |
            ((cov >= self.resolver_cov_national_min) & (ent >= self.resolver_ent_national_min))
        )

        # Big-DMA corner targeting (again, in case it wasn't applied or mixed ads still exists)
        corner_national = (
            (ent <= self.corner_entropy_max) &
            (cov <= self.corner_coverage_max) &
            (reg90 >= self.corner_region_min)
        )

        # Apply only to mixed ads
        to_local = m_border & local_resolver
        to_nat = m_border & (national_resolver | corner_national)

        res.loc[to_local, "label"] = "local"
        res.loc[to_local, "confidence"] = "high"
        res.loc[to_local, "override_reason"] = self._append_reason(res.loc[to_local, "override_reason"], "mixed_ad_resolve_local")

        res.loc[to_nat, "label"] = "national"
        res.loc[to_nat, "confidence"] = "high"
        res.loc[to_nat, "override_reason"] = self._append_reason(res.loc[to_nat, "override_reason"], "mixed_ad_resolve_national")

        return res

    # ----------------------------
    # Main entry
    # ----------------------------
    def predict(self, df: pd.DataFrame) -> pd.DataFrame:
        self.validate_columns(df)
        out = df.copy()

        # localness
        out["localness"] = self.compute_localness(out)

        # calibration
        if self.iso is None:
            self.fit_isotonic(out, score_col="localness")

        out["p_local_cal"] = self.iso.predict(out["localness"].to_numpy())
        out["p_local_cal"] = pd.to_numeric(out["p_local_cal"], errors="coerce").fillna(0.5).clip(0, 1)

        # overrides
        out["override_reason"] = ""
        out = self.apply_overrides(out)

        # base labels
        out["label"] = np.where(out["p_local_cal"] >= self.p_hi, "local",
                         np.where(out["p_local_cal"] <= self.p_lo, "national", "mixed"))
        out["confidence"] = np.where(
            (out["p_local_cal"] >= self.p_hi) | (out["p_local_cal"] <= self.p_lo),
            "high", "low"
        )

        # resolve mixed ads to reach <10% without relaxing p thresholds much
        out = self.resolve_borderline(out)

        # recompute p_national (in case overrides changed p_local)
        out["p_national_cal"] = 1 - out["p_local_cal"]

        return out

In [0]:
clf = LocalNationalCalibrator(
    corner_region_min=4,
    p_lo=0.17,
    p_hi=0.83,
    seed_lo=0.17,
    seed_hi=0.83,
    min_imps_local_override=20,
)

In [0]:
scored = clf.predict(df)

In [0]:
scored["label"].value_counts(dropna=False)

In [0]:
scored.head().display()

In [0]:
ads = ['AE16546-2024-43-05904' ,'AE16546-2022-47-00099' ,'AE16546-2024-21-06012' ,'AE16546-2024-17-05047' ,'AE16546-2024-22-00694' ,'AE16546-2024-22-01098' ,'AE16546-2024-22-01123' ,'AE16546-2026-02-00621' ,'AE16546-2024-16-07265' ,'AE16546-2026-03-06616' ,'AE16546-2023-48-03914' ,'AE16546-2023-50-04726' ,'AE16546-2025-05-00291' ,'AE16546-2025-05-05606' ,'AE16546-2026-04-03160' ,'AE16546-2026-04-03563' ,'AE16546-2025-39-01643' ,'AE16546-2023-36-04733' ,'AE16546-2023-36-05942' ,'AE16546-2024-19-09078' ,'AE16546-2024-17-03578' ,'AE16546-2024-18-05170' ,'AE16546-2024-12-07052' ,'AE16546-2023-40-03098' ,'AE16546-2023-49-09544' ,'AE16546-2024-18-04728' ,'AE16546-2024-18-05181' ,'AE16546-2024-18-06023' ,'AE16546-2024-18-06189' ,'AE16546-2024-18-06899' ,'AE16546-2024-18-06941' ,'AE16546-2024-18-07835' ,'AE16546-2024-19-02259' ,'AE16546-2024-21-04910' ,'AE16546-2024-23-04445' ,'AE16546-2025-24-03712' ,'AE16546-2021-14-05515' ,'AE16546-2023-36-09272' ,'AE16546-2023-48-04122' ,'AE16546-2023-48-06997' ,'AE16546-2025-02-01764' ,'AE16546-2025-10-08274' ,'AE16546-2023-29-00452' ,'AE16546-2024-07-03058' ,'AE16546-2023-36-06040' ,'AE16546-2023-49-10589' ,'AE16546-2024-17-00026' ,'AE16546-2023-12-06109' ,'AE16546-2022-30-07282' ,'AE16546-2024-31-04707' ,'AE16546-2023-10-01124' ,'AE16546-2021-40-09374' ,'AE16546-2021-41-00647' ,'AE16546-2026-02-01242' ,'AE16546-2025-22-00599' ,'AE16546-2023-48-03459' ,'AE16546-2023-42-01550' ,'AE16546-2023-28-07250' ,'AE16546-2023-37-07508' ,'AE16546-2023-48-05065' ,'AE16546-2023-50-03279' ,'AE16546-2024-16-08648' ,'AE16546-2024-21-00513'
]
scored[scored.ad_id.isin(ads)].display()

In [0]:
scored.label.value_counts()

In [0]:
def plot_kde(df: pd.DataFrame, x: str, y: str, x_name: str, y_name: str):
    sns.kdeplot(
        data=df,
        x=x,
        y=y,
        levels=100,
        thresh=0.02,
        fill=False,
        alpha=0.4,
    )

    plt.title(f"KDE Plot - {x_name} vs {y_name}")
    plt.xlabel(x_name)
    plt.ylabel(y_name)
    plt.show()

In [0]:
plot_dict = {
    "coverage_score": "Coverage Score",
    "entropy_norm": "Entropy Norm",
    "mix_ratio": "Top 5 DMA Mix Ratio",
    "sig_dma_count": "Significant DMAs",
    "dma_in_95": "# of DMAs in 90 percent of Ads",
}
done = []
for x, v1 in plot_dict.items():
    for y, v2 in plot_dict.items():
        if x != y and sorted([x, y]) not in done:
            try:
                done.append(sorted([x, y]))
                plot_kde(df=ndf, x=x, y=y, x_name=v1, y_name=v2)
            except ValueError:
                print(f'failed {x} {y}')

In [0]:
def add_localness_score(df: pd.DataFrame) -> pd.DataFrame:
    # Convert to "local direction"
    cov_local = 1.0 - df["coverage_score"]
    ent_local = 1.0 - df["entropy_norm"]
    mix_local = df["mix_ratio"]
    mix_region = df["norm_reg_mix"]

    # Weighted average (entropy very important, mix very important, coverage important)
    df.loc[:, "localness_score"] = 0.35 * mix_local + 0.35 * ent_local + 0.20 * cov_local + 0.1 * mix_region
    return df

ndf = add_localness_score(ndf)
# df = add_localness_score(df)

In [0]:
if 'localness_score' not in metrics: metrics.append('localness_score')

In [0]:
# qs = [0.01, 0.10, 0.15, 0.20, 0.25, 0.50, 0.75, 0.80, 0.85, 0.90, 0.95, 0.99]
# quantiles = ndf[metrics].quantile(qs)
# quantiles.display()
# cov = quantiles["coverage_score"].to_list()
# ent = quantiles["entropy_norm"].to_list()
# mix = quantiles["mix_ratio"].to_list()
# sig = quantiles["sig_dma_count"].to_list()
# lcs = quantiles["localness_score"].to_list()
# d95 = quantiles["dma_in_95"].to_list()

# cov = list(zip(qs, cov))
# ent = list(zip(qs, ent))
# mix = list(zip(qs, mix))
# sig = list(zip(qs, sig))
# lcs = list(zip(qs, lcs))
# d95 = list(zip(qs, d95))

# ndf.loc[:, 'coverage_quant'] = 'p1'
# ndf.loc[:, 'entropy_quant'] = 'p1'
# ndf.loc[:, 'mix_quant'] = 'p1'
# ndf.loc[:, 'sigdma_quant'] = 'p1'
# ndf.loc[:, 'localness_quant'] = 'p1'
# ndf.loc[:, 'd95_quant'] = 'p1'

# for q, i in cov:
#     ndf.loc[:, 'coverage_quant'] = np.where(ndf.coverage_score >= i, f'p{int(q*100)}', ndf.coverage_quant)

# for q, i in ent:
#     ndf.loc[:, 'entropy_quant'] = np.where(ndf.entropy_norm >= i, f'p{int(q*100)}', ndf.entropy_quant)

# for q, i in mix:
#     ndf.loc[:, 'mix_quant'] = np.where(ndf.mix_ratio >= i, f'p{int(q*100)}', ndf.mix_quant)

# for q, i in sig:
#     ndf.loc[:, 'sigdma_quant'] = np.where(ndf.sig_dma_count >= i, f'p{int(q*100)}', ndf.sigdma_quant)

# for q, i in lcs:
#     ndf.loc[:, 'localness_quant'] = np.where(ndf.localness_score >= i, f'p{int(q*100)}', ndf.localness_quant)

# for q, i in d95:
#     ndf.loc[:, 'd95_quant'] = np.where(ndf.dma_in_95 >= i, f'p{int(q*100)}', ndf.d95_quant)

# ndf.head(50).display()

In [0]:
def create_range(l: list, sing=False) -> list:
    f = []
    for i in range(1, len(l)):
        try:
            if sing == False:
                f.append(f"`{l[i]}-{l[i+1]}")
            else:
                if i < len(l) - 1:
                    if l[i] < 10:
                        f.append(f"`0{l[i]}")
                    else:
                        f.append(f"`0{l[i]}")
                else:
                    if l[i] < 10:
                        f.append(f"`0{l[i]}+")
                    else:
                        f.append(f"`0{l[i]}+")
        except IndexError:
            f.append(f"`{l[i]}+")
    return list(zip(f, l))

In [0]:
def add_bin_columns(df: pd.DataFrame, min_: str, rng: list, new_col: str, col: str):
    df.loc[:, new_col] = min_
    for q, i in rng:
        df.loc[:, new_col] = np.where(df[col] >= i, q, df[col])
    return df

In [0]:
# cov = [0.0095, 0.0145, 0.1185, 0.2370, 0.4740, 0.7109]
# ent = [0.10, 0.15, 0.25, 0.50, 0.60, 0.75, 0.80]
# mix = [0.20, 0.225, 0.25, 0.50, 0.90, 0.925, 0.95, 0.975]
# sig = [1,2,3,4,5,6]
# lcs = [0.20, 0.30, 0.40, 0.45, 0.50, 0.55, 0.60, 0.70, 0.80]
# d95 = [2,3,4,5,6,10,20,50,100]

# cov = create_range(l=cov, sing=False)
# ent = create_range(l=ent, sing=False)
# mix = create_range(l=mix, sing=False)
# sig = create_range(l=sig, sing=True)
# lcs = create_range(l=lcs, sing=False)
# d95 = create_range(l=d95, sing=False)

# ndf.loc[:, 'coverage_quant'] = '<=0.0095'
# ndf.loc[:, 'entropy_quant'] = '<=0.10'
# ndf.loc[:, 'mix_quant'] = '<=0.20'
# ndf.loc[:, 'sigdma_quant'] = '0'
# ndf.loc[:, 'localness_quant'] = '<=0.20'
# ndf.loc[:, 'd95_quant'] = '`<=2'

# for q, i in cov:
#     ndf.loc[:, 'coverage_quant'] = np.where(ndf.coverage_score > i, q, ndf.coverage_quant)

# for q, i in ent:
#     ndf.loc[:, 'entropy_quant'] = np.where(ndf.entropy_norm > i, q, ndf.entropy_quant)

# for q, i in mix:
#     ndf.loc[:, 'mix_quant'] = np.where(ndf.mix_ratio > i, q, ndf.mix_quant)

# for q, i in sig:
#     ndf.loc[:, 'sigdma_quant'] = np.where(ndf.sig_dma_count > i, q, ndf.sigdma_quant)

# for q, i in lcs:
#     ndf.loc[:, 'localness_quant'] = np.where(ndf.localness_score > i, q, ndf.localness_quant)

# for q, i in d95:
#     ndf.loc[:, 'd95_quant'] = np.where(ndf.dma_in_95 > i, q, ndf.d95_quant)

In [0]:
# ndf.groupby(
#     [
#         # "coverage_quant",
#         "entropy_quant",
#         # "mix_quant",
#         # "sigdma_quant",
#         # "d95_quant",
#         "localness_quant",
#     ]
# ).ad_id.count().reset_index().display()

In [0]:
# ndf[ndf.localness_score <= 0.15].entropy_norm.hist()
# ndf[ndf.localness_score <= 0.15].coverage_score.hist()
# ndf[ndf.localness_score <= 0.15].mix_ratio.hist()
# ndf[ndf.localness_score <= 0.25].sig_dma_count.hist(bins=20)
# ndf[ndf.localness_score <= 0.25].entropy_norm.quantile([0.01, 0.10, 0.25, 0.50, 0.75, 0.90, 0.99]).to_dict()
# ndf[ndf.localness_score <= 0.25].coverage_score.quantile([0.01, 0.10, 0.25, 0.50, 0.75, 0.90, 0.99]).to_dict()
# ndf[ndf.localness_score <= 0.25].mix_ratio.quantile([0.01, 0.10, 0.25, 0.50, 0.75, 0.90, 0.99]).to_dict()
ndf[ndf.mix_ratio <= 0.25].dma_in_95.quantile([0.01, 0.10, 0.25, 0.50, 0.75, 0.90, 0.99])

In [0]:
# ndf[ndf.localness_score >= 0.90].entropy_norm.hist()
# ndf[ndf.localness_score >= 0.75].coverage_score.hist()
# ndf[ndf.localness_score >= 0.75].mix_ratio.hist()
# ndf[ndf.localness_score >= 0.75].sig_dma_count.hist()

# ndf[ndf.localness_score >= 0.75].entropy_norm.quantile([0.01, 0.10, 0.25, 0.50, 0.75, 0.90, 0.99]).to_dict()
# ndf[ndf.localness_score >= 0.75].coverage_score.quantile([0.01, 0.10, 0.25, 0.50, 0.75, 0.90, 0.99]).to_dict()
# ndf[ndf.localness_score >= 0.75].mix_ratio.quantile([0.01, 0.10, 0.25, 0.50, 0.75, 0.90, 0.99]).to_dict()
# ndf[ndf.localness_score >= 0.75].sig_dma_count.quantile([0.01, 0.10, 0.25, 0.50, 0.75, 0.90, 0.99]).to_dict()
# ndf[ndf.dma_in_95 >= 120][metrics].quantile([0.00001, 0.10, 0.25, 0.50, 0.75, 0.90, 0.999999]).display()
# ndf[ndf.dma_in_95 >= 1    00][metrics].quantile([0.00001, 0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.99999]).display()
ndf[ndf.localness_score <= 0.15][metrics].quantile([0.10, 0.20, 0.80, 0.90]).display()
ndf[ndf.localness_score >= 0.85][metrics].quantile([0.10, 0.20, 0.80, 0.90]).display()

In [0]:
ndf[(ndf.sig_dma_count == 1) & (ndf.localness_score <= 0.40)].display()

In [0]:
ads = ['AE16546-2024-43-05904' ,'AE16546-2022-47-00099' ,'AE16546-2024-21-06012' ,'AE16546-2024-17-05047' ,'AE16546-2024-22-00694' ,'AE16546-2024-22-01098' ,'AE16546-2024-22-01123' ,'AE16546-2026-02-00621' ,'AE16546-2024-16-07265' ,'AE16546-2026-03-06616' ,'AE16546-2023-48-03914' ,'AE16546-2023-50-04726' ,'AE16546-2025-05-00291' ,'AE16546-2025-05-05606' ,'AE16546-2026-04-03160' ,'AE16546-2026-04-03563' ,'AE16546-2025-39-01643' ,'AE16546-2023-36-04733' ,'AE16546-2023-36-05942' ,'AE16546-2024-19-09078' ,'AE16546-2024-17-03578' ,'AE16546-2024-18-05170' ,'AE16546-2024-12-07052' ,'AE16546-2023-40-03098' ,'AE16546-2023-49-09544' ,'AE16546-2024-18-04728' ,'AE16546-2024-18-05181' ,'AE16546-2024-18-06023' ,'AE16546-2024-18-06189' ,'AE16546-2024-18-06899' ,'AE16546-2024-18-06941' ,'AE16546-2024-18-07835' ,'AE16546-2024-19-02259' ,'AE16546-2024-21-04910' ,'AE16546-2024-23-04445' ,'AE16546-2025-24-03712' ,'AE16546-2021-14-05515' ,'AE16546-2023-36-09272' ,'AE16546-2023-48-04122' ,'AE16546-2023-48-06997' ,'AE16546-2025-02-01764' ,'AE16546-2025-10-08274' ,'AE16546-2023-29-00452' ,'AE16546-2024-07-03058' ,'AE16546-2023-36-06040' ,'AE16546-2023-49-10589' ,'AE16546-2024-17-00026' ,'AE16546-2023-12-06109' ,'AE16546-2022-30-07282' ,'AE16546-2024-31-04707' ,'AE16546-2023-10-01124' ,'AE16546-2021-40-09374' ,'AE16546-2021-41-00647' ,'AE16546-2026-02-01242' ,'AE16546-2025-22-00599' ,'AE16546-2023-48-03459' ,'AE16546-2023-42-01550' ,'AE16546-2023-28-07250' ,'AE16546-2023-37-07508' ,'AE16546-2023-48-05065' ,'AE16546-2023-50-03279' ,'AE16546-2024-16-08648' ,'AE16546-2024-21-00513'
]
ndf[ndf.ad_id.isin(ads)].display()

In [0]:
ent_nat_min = ndf[ndf.localness_score <= 0.20].entropy_norm.quantile(0.10)
cov_nat_min = ndf[ndf.localness_score <= 0.20].coverage_score.quantile(0.10)
mix_nat_max = ndf[ndf.localness_score <= 0.20].mix_ratio.quantile(0.90)
d90_nat_min = ndf[ndf.localness_score <= 0.20].dma_in_90.quantile(0.10)
rmx_nat_max = ndf[ndf.localness_score <= 0.20].region_mix.quantile(0.90)

ent_loc_max = ndf[ndf.localness_score >= 0.80].entropy_norm.quantile(0.90)
cov_loc_max = ndf[ndf.localness_score >= 0.80].coverage_score.quantile(0.90)
mix_loc_min = ndf[ndf.localness_score >= 0.80].mix_ratio.quantile(0.10)
d90_loc_max = ndf[ndf.localness_score >= 0.80].dma_in_90.quantile(0.90)
rmx_loc_min = ndf[ndf.localness_score >= 0.80].region_mix.quantile(0.10)

In [0]:
initital_cutoffs = {
    "coverage_national": cov_nat_min,
    "entropy_national": ent_nat_min,
    "mix_national": mix_nat_max,
    "d90_national": d90_nat_min,
    "rmx_national": rmx_nat_max,
    "coverage_local": cov_loc_max,
    "entropy_local": ent_loc_max,
    "mix_local": mix_loc_min,
    "d90_local": d90_loc_max,
    "rmx_local": rmx_loc_min
}
initital_cutoffs

In [0]:
def classify_ads_v1(row):
    cov = row["coverage_score"]
    ent = row["entropy_norm"]
    mix = row["mix_ratio"]
    d90 = row["dma_in_90"]
    loc = row["localness_score"]
    rmx = row["region_mix"]
    r90 = row["region_count_90"]
    
    nat_conds = [
        cov >= initital_cutoffs['coverage_national'],
        ent >= initital_cutoffs['entropy_national'],
        mix <= initital_cutoffs['mix_national'],
        # d90 >= initital_cutoffs['d90_national'],
        rmx <= initital_cutoffs['rmx_national'],
    ]
    nat_score = sum(nat_conds)

    loc_conds = [
        cov <= initital_cutoffs['coverage_local'],
        ent <= initital_cutoffs['entropy_local'],
        mix >= initital_cutoffs['mix_local'],
        # d90 <= initital_cutoffs['d90_local'],
        rmx >= initital_cutoffs['rmx_local'],
    ]
    loc_score = sum(loc_conds)

    if d90 == 1 or r90 == 1:
        return "Local"
    elif loc_score >= 3:
        return "Local"
    elif nat_score >= 3:
        return "National"
    elif loc >= 0.80:
        return "Local"
    elif loc <= 0.20:
        return "National"
    # elif rmx <= 0.30:
    #     return "National"
    # elif rmx >= 0.90:
    #     return "Local"

    return 'Mixed'

In [0]:
ndf["ad_type"] = ndf.apply(classify_ads_v1, axis=1)
# df["ad_type"] = df.apply(classify_ads_v1, axis=1)

nat = ndf[ndf["ad_type"] == "National"]
loc = ndf[ndf["ad_type"] == "Local"]
mix = ndf[ndf["ad_type"] == "Mixed"]
len(nat), len(loc), len(mix)

In [0]:
def print_seed_quantiles(nat_seed, loc_seed):
    cols = metrics
    nat_seed_float = nat_seed[cols].astype(float)
    loc_seed_float = loc_seed[cols].astype(float)
    # mix_seed_float = mix_seed[cols].astype(float)

    print("=== National seed quantiles ===")
    quant = [i/20 for i in range(1, 20)]
    # display(nat_seed_float.quantile([0.05, 0.10, 0.25, 0.5, 0.75, 0.90, 0.95]))
    quant.append(0.999)
    x = nat_seed_float.quantile(quant)
    x.reset_index(inplace=True)
    display(x)

    print("\n=== Local seed quantiles ===")
    x = loc_seed_float.quantile(quant)
    x.reset_index(inplace=True)
    display(x)

    # if mix_seed is not None:
    #     mix_seed_float = mix_seed[cols].astype(float)
    #     print("\n=== Mixed seed quantiles ===")
    #     display(mix_seed_float.quantile(quant))

print_seed_quantiles(nat, loc)

In [0]:
second_cutoffs = {
    "coverage_national": nat.coverage_score.quantile(0.10),
    "entropy_national": nat.entropy_norm.quantile(0.10),
    "mix_national": nat.mix_ratio.quantile(0.90),
    "localness_national": nat.localness_score.quantile(0.90),
    "d90_national": nat.dma_in_90.quantile(0.10),
    "rmx_national": nat.region_mix.quantile(0.90),
    "coverage_local": loc.coverage_score.quantile(0.90),
    "entropy_local": loc.entropy_norm.quantile(0.90),
    "mix_local": loc.mix_ratio.quantile(0.10),
    "localness_local": loc.localness_score.quantile(0.10),
    "d90_local": loc.dma_in_90.quantile(0.90),
    "rmx_local": loc.region_mix.quantile(0.10),
}
second_cutoffs

In [0]:
def classify_ads_v2(row):
    if row["ad_type"] == "National":
        return "National"
    elif row["ad_type"] == "Local":
        return "Local"

    cov = row["coverage_score"]
    ent = row["entropy_norm"]
    mix = row["mix_ratio"]
    d90 = row["dma_in_90"]
    loc = row["localness_score"]
    sig = row["sig_dma_count"]
    rmx = row["region_mix"]

    cov_nat = second_cutoffs['coverage_national']
    ent_nat = second_cutoffs['entropy_national']
    mix_nat = second_cutoffs['mix_national']
    d90_nat = second_cutoffs['d90_national']
    loc_nat = second_cutoffs['localness_national']
    rmx_nat = second_cutoffs['rmx_national']

    cov_loc = second_cutoffs['coverage_local']
    ent_loc = second_cutoffs['entropy_local']
    mix_loc = second_cutoffs['mix_local']
    d90_loc = second_cutoffs['d90_local']
    loc_loc = second_cutoffs['localness_local']
    rmx_loc = second_cutoffs['rmx_local']
    
    nat_conds = [
        cov >= cov_nat,
        ent >= ent_nat,
        mix <= mix_nat,
        # d90 >= d90_nat,
        loc <= loc_nat,
        rmx <= rmx_nat,
    ]
    nat_score = sum(nat_conds)

    loc_conds = [
        cov <= cov_loc,
        ent <= ent_loc,
        mix >= mix_loc,
        # d90 <= d90_loc,
        loc >= loc_loc,
        rmx >= rmx_loc,
    ]
    loc_score = sum(loc_conds)
    # ----- HARDCODED COVERAGE OVERRIDES -----
    if loc_score >= 4:
        return "Local"
    elif nat_score >= 4:
        return "National"
    elif loc <= loc_nat:
        return "National"
    elif loc >= loc_loc:
        return "Local"
    elif loc_score >= 3 and loc_score > nat_score:
        return "Local"
    elif nat_score >= 3 and nat_score > loc_score:
        return "National"
    elif mix <= mix_nat and d90 >= d90_nat:
        return "National"
    elif mix >= mix_loc and d90 <= d90_loc:
        return "Local"
    elif sig == 0 and mix <= mix_nat:
        return "National"
    elif ent >= ent_nat:
        return "National"
    elif cov >= cov_nat and ent >= ent_nat:
        return "National"
    elif cov <= cov_loc and ent <= ent_loc:
        return "Local"
    elif mix >= mix_loc and cov <= cov_loc:
        return "Local"
    # elif ent <= 0.15:
    #     return "Local"
    # elif d90 <= 2:
    #     return "Local"
    # elif d90 >= 100:
    #     return "National"
    # elif loc > 0.50:
    #     return "Local"
    # elif loc <= 0.50:
    #     return "National"
    return 'Mixed'

In [0]:
ndf["ad_type_v2"] = ndf.apply(classify_ads_v2, axis=1)

nat = ndf[ndf["ad_type_v2"] == "National"]
loc = ndf[ndf["ad_type_v2"] == "Local"]
mix = ndf[ndf["ad_type_v2"] == "Mixed"]
len(nat), len(loc), len(mix)

In [0]:
df["ad_type_final"] = df.apply(classify_ads_v2, axis=1)


In [0]:
len(df[df["ad_type_final"] == "National"]), len(df[df["ad_type_final"] == "Local"]), len(df[df["ad_type_final"] == "Mixed"])

In [0]:
df[df.ad_type_final == 'National'].entropy_norm.hist(bins=50)

In [0]:
cols = [
    "coverage_score",
    "entropy_norm",
    "mix_ratio",
    'localness_score',
    'dma_in_95'
]
nat_seed_float = ndf[ndf.ad_type_v2 == 'National'][cols].astype(float)
loc_seed_float = ndf[ndf.ad_type_v2 == 'Local'][cols].astype(float)

print("=== National seed quantiles ===")
quant = [i/20 for i in range(1, 20)]
quant.append(0.99)
# display(nat_seed_float.quantile([0.05, 0.10, 0.25, 0.5, 0.75, 0.90, 0.95]))
display(nat_seed_float.quantile(quant))

print("\n=== Local seed quantiles ===")
display(loc_seed_float.quantile(quant))

In [0]:
def plot_a_vs_b(df_plot, a, b, a_name, b_name):
    plt.figure(figsize=(8, 7))

    for t in df_plot['ad_type_v2'].unique():
        subset = df_plot[df_plot['ad_type_v2'] == t]
        plt.scatter(
            subset[a],
            subset[b],
            alpha=0.25,
            label=t,
            s=2,
        )

    plt.ylabel(b_name)
    plt.xlabel(a_name)
    plt.title(f'{a_name} vs {b_name}')
    plt.legend()
    plt.show()

In [0]:

plot_dict = {
    "coverage_score": [0.020, 0.057, "Coverage Score"],
    "entropy_norm": [0.794, 0.609, "Entropy Norm"],
    "mix_ratio": [2.96, 3.314, "DMA Mix Ratio"],
    "sig_dma_count": [0.020, 0.057, "Sig DMA Count"],
    "dma_in_90": [0.020, 0.057, "DMAs in 90%"],
    "localness_score": [5.528, 2.155, "Localness Score"],
    "total_states": [5.528, 2.155, "Number of US States"],
}
done = []
for a, v1 in plot_dict.items():
    for b, v2 in plot_dict.items():
        if a != b and sorted([a, b]) not in done:
            done.append(sorted([a, b]))
            plot_a_vs_b(
                df_plot=ndf,
                a=a,
                b=b,
                a_name=v1[2],
                b_name=v2[2],
            )

In [0]:
def plot_kde(df: pd.DataFrame, x: str, y: str, x_name: str, y_name: str):
    sns.kdeplot(
        data=df,
        x=x,
        y=y,
        hue='ad_type_v2',
        levels=100,
        thresh=0.02,
        fill=False,
        alpha=0.4,
    )

    plt.title(f"KDE Plot - {x_name} vs {y_name}")
    plt.xlabel(x_name)
    plt.ylabel(y_name)
    plt.show()

In [0]:
plot_dict = {
    "coverage_score": "Coverage Score",
    "entropy_norm": "Entropy Norm",
    "mix_ratio": "DMA Mix Ratio",
    "localness_score": "Localness Score",
    "dma_in_90": "DMAs in 90 Percent of Ads",
}
done = []
for x, v1 in plot_dict.items():
    for y, v2 in plot_dict.items():
        if x != y and sorted([x, y]) not in done:
            try:
                done.append(sorted([x, y]))
                plot_kde(df=ndf, x=x, y=y, x_name=v1, y_name=v2)
            except ValueError:
                print(f'failed {x} {y}' )

In [0]:
def plot_boxplot_by_label(df, col, label_col="ad_type_v2"):
    data = [
        df[df[label_col] == "Local"][col].dropna(),
        df[df[label_col] == "National"][col].dropna()
        ]
    labels = ["Local", "National"]

    plt.figure(figsize=(6, 5))
    plt.boxplot(data, labels=labels, showfliers=False)
    plt.title(f"{col} distribution by label")
    plt.ylabel(col)
    plt.xlabel("ad_type")
    plt.grid(True, axis="y")
    plt.show()

In [0]:
def plot_hist_by_label(df, col, label_col="ad_type_v2", bins=50):
    plt.figure(figsize=(7, 5))
    for label in ["Local", "National"]:
        subset = df[df[label_col] == label][col].dropna()
        plt.hist(subset, bins=bins, alpha=0.5, label=label)

    plt.xlabel(col)
    plt.ylabel("Count")
    plt.title(f"{col} histogram by label")
    plt.legend()
    plt.grid(True)
    plt.show()

In [0]:
for col in ["coverage_score", "entropy_norm", "mix_ratio"]:
    plot_boxplot_by_label(ndf, col)

In [0]:
for col in ["coverage_score", "entropy_norm", "mix_ratio"]:
    plot_hist_by_label(ndf, col)

In [0]:
df.columns

In [0]:
final_df = scored[
    [
        "ad_id",
        "brand",
        "total_impressions",
        "total_opportunities",
        "coverage_score",
        "entropy_norm",
        "sig_dma_count",
        "dma_in_90",
        "mix_ratio",
        "region_count_90",
        "sig_region_count",
        "region_mix",
        "norm_reg_mix",
        "localness",
        "p_local_cal",
        "override_reason",
        "label",
        "confidence",
    ]
].copy()

In [0]:
final_df = df[
    [
        "ad_id",
        "total_impressions",
        "total_opportunities",
        "coverage_score",
        "entropy_norm",
        "mix_ratio",
        "sig_dma_count",
        "localness_score",
        "dma_in_90",
        "ad_type_final",
    ]
].copy()

In [0]:
ndf[ndf.ad_id == 'AE16546-2025-50-03283'].display()

In [0]:
final_df.head().display()

In [0]:
for col in ['coverage_score', 'entropy_norm', 'sig_dma_count', 'dma_in_90', 'mix_ratio', 'region_count_90', 'sig_region_count', 'region_mix', 'norm_reg_mix', 'localness', 'p_local_cal']:
    final_df.loc[:, col] = final_df[col].astype(float)

In [0]:
final_df.info()

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_labeling_national_local_set_012126;

In [0]:
spark_df = spark.createDataFrame(final_df)
spark_df.write.option("mergeSchema", "true").mode("overwrite").saveAsTable("dev.mohit_gangwani.ad_labeling_national_local_set_012126")


In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_labeling_national_local_set_12_16_25_new;

In [0]:
spark_df.write.option("mergeSchema", "true").mode("overwrite").saveAsTable("dev.mohit_gangwani.ad_labeling_national_local_set_12_16_25_new")